# Campaign Planning, Simplified: Build It with CoCo, Ask It with CoWork

**Meridian Stay** is a global hotel brand. Its regional and channel teams plan campaigns in three different systems — ad platforms, an email & SMS platform, and the CRM — and nobody has a single view of what's launching. Last quarter, two teams hit the same audience with competing offers in the same week.

In this lab you'll fix that:

1. **Land** the raw campaign exports in open **Apache Iceberg** tables.
2. **Clean and standardize** them with **CoCo**, and find every audience collision.
3. **See** the result in a live **campaign calendar** app.
4. **Ask** questions about it in plain language with **Snowflake CoWork** (done in the Snowsight UI — see the guide).

Some cells are pre-written; others you'll generate by prompting **CoCo** (the prompt is in the markdown cell right above each one).

**Schemas:** `RAW` (exports as they arrived) → `CURATED` (clean, governed tables) → `ANALYTICS` (semantic view and agent).

## Connect this notebook to compute

This notebook runs inside a **Git-backed Workspace**. Before running any cell:

1. Click **Connect** next to the Run button and select **Create and connect**. Wait for the status bar at the bottom of the notebook to show **Connected**.

2. Set your **role** and **warehouse** using the picker at the top of the Notebooks editor:
   - **Role = `ACCOUNTADMIN`**
   - **Warehouse = `COMPUTE_WH`** (the default trial warehouse)

3. Notebooks in Workspaces **don't auto-select a database or schema**, so every cell below uses fully qualified names (`MERIDIAN_STAY.RAW.…`).

4. Run cells top-to-bottom with **▶** / **Shift+Enter**. If the session idle-suspends, restart it and re-run from where you left off — objects you created persist in Snowflake.

Where a markdown cell shows a **CoCo prompt**, open the **CoCo** panel, paste the prompt, compare CoCo's output against the **Expected output** in the cell, and click **Allow** to run it. Optionally, copy the SQL into the empty cell below the prompt for future reference.

In [ ]:
-- SETUP: create the Meridian Stay database and grant the Cortex privileges used later in CoWork.
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

CREATE OR REPLACE DATABASE MERIDIAN_STAY;
CREATE OR REPLACE SCHEMA MERIDIAN_STAY.RAW;
CREATE OR REPLACE SCHEMA MERIDIAN_STAY.CURATED;
CREATE OR REPLACE SCHEMA MERIDIAN_STAY.ANALYTICS;

-- Privileges needed later to create the Cortex Agent and use CoWork
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE ACCOUNTADMIN;
ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION';

## Land the raw exports in Apache Iceberg tables

The three campaign exports live in this repo's `data/` folder:

| File | Source system | How it's messy |
|---|---|---|
| `paid_media_export.csv` | Ad platforms | Channel codes like `fb_ads`, budgets stored as text (`"$65,000"`), re-exported duplicate rows |
| `email_sms_export.csv` | Email & SMS platform (e.g., Marketo, HubSpot, Braze) | `MM/DD/YYYY` dates, its own region and segment names |
| `crm_campaigns_export.csv` | CRM | `Oct 05, 2026` dates, copies of the big paid campaigns, a **cancelled** campaign still in the list |

We land each one, exactly as it arrived, in an **Apache Iceberg** table. Iceberg is an open table format: the data is stored as Parquet files plus open metadata, so other engines (Spark, Trino, DuckDB, …) can read the same tables through Snowflake Horizon Catalog. Your campaign data stays open, not locked into one tool.

`CATALOG = 'SNOWFLAKE'` + `EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'` means Snowflake manages the Iceberg storage for you — no cloud bucket or IAM setup needed.

Run the next cell as-is. It creates the three raw Iceberg tables (every column as `STRING`, exactly as exported), a CSV file format, and an internal stage for the files.

In [ ]:
CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY.RAW.PAID_MEDIA_EXPORT (
    ad_campaign_id   STRING,
    campaign_name    STRING,
    platform_channel STRING,
    geo              STRING,
    target_audience  STRING,
    flight_start     STRING,
    flight_end       STRING,
    budget           STRING,
    owner            STRING
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED';

CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY.RAW.EMAIL_SMS_EXPORT (
    program_id    STRING,
    program_name  STRING,
    channel_type  STRING,
    region_name   STRING,
    segment       STRING,
    send_start    STRING,
    send_end      STRING,
    budget_usd    STRING,
    team          STRING
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED';

CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY.RAW.CRM_CAMPAIGNS_EXPORT (
    crm_campaign_id STRING,
    campaign_title  STRING,
    campaign_type   STRING,
    territory       STRING,
    audience_group  STRING,
    start_date      STRING,
    end_date        STRING,
    budgeted_cost   STRING,
    status          STRING,
    source_system   STRING
)
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED';

CREATE OR REPLACE FILE FORMAT MERIDIAN_STAY.RAW.CSV_FF
  TYPE = 'CSV'
  SKIP_HEADER = 1
  FIELD_OPTIONALLY_ENCLOSED_BY = '"';

CREATE OR REPLACE STAGE MERIDIAN_STAY.RAW.CAMPAIGN_EXPORTS
  FILE_FORMAT = MERIDIAN_STAY.RAW.CSV_FF;

### Upload the export files to the stage

This Python cell copies the three CSVs from the repo's `data/` folder into the `CAMPAIGN_EXPORTS` stage. Run it as-is — you should see three files with status `UPLOADED`.

In [ ]:
import os
from snowflake.snowpark.context import get_active_session

session = get_active_session()

for f in ["paid_media_export.csv", "email_sms_export.csv", "crm_campaigns_export.csv"]:
    result = session.file.put(
        os.path.abspath(os.path.join("data", f)),
        "@MERIDIAN_STAY.RAW.CAMPAIGN_EXPORTS",
        auto_compress=False,
        overwrite=True,
    )
    print(result[0].source, result[0].status)

### STEP 1 — Load the exports into the raw Iceberg tables

`COPY INTO` is Snowflake's bulk-loading command — it reads files from a stage and inserts them into a table. It works the same way whether the target is a standard table or an Iceberg table.

Open the **CoCo** panel and send this prompt. Compare CoCo's output against the expected output below — if they match, click **Allow** to run it.

> *"Load the three CSV files in the stage @MERIDIAN_STAY.RAW.CAMPAIGN_EXPORTS into their matching Iceberg tables in MERIDIAN_STAY.RAW: paid_media_export.csv into PAID_MEDIA_EXPORT, email_sms_export.csv into EMAIL_SMS_EXPORT, and crm_campaigns_export.csv into CRM_CAMPAIGNS_EXPORT. Use the file format MERIDIAN_STAY.RAW.CSV_FF."*

You should see **16**, **10**, and **7** rows loaded.

<details>
<summary>Expected output (check against this)</summary>

```sql
COPY INTO MERIDIAN_STAY.RAW.PAID_MEDIA_EXPORT
  FROM @MERIDIAN_STAY.RAW.CAMPAIGN_EXPORTS/paid_media_export.csv
  FILE_FORMAT = (FORMAT_NAME = MERIDIAN_STAY.RAW.CSV_FF);

COPY INTO MERIDIAN_STAY.RAW.EMAIL_SMS_EXPORT
  FROM @MERIDIAN_STAY.RAW.CAMPAIGN_EXPORTS/email_sms_export.csv
  FILE_FORMAT = (FORMAT_NAME = MERIDIAN_STAY.RAW.CSV_FF);

COPY INTO MERIDIAN_STAY.RAW.CRM_CAMPAIGNS_EXPORT
  FROM @MERIDIAN_STAY.RAW.CAMPAIGN_EXPORTS/crm_campaigns_export.csv
  FILE_FORMAT = (FORMAT_NAME = MERIDIAN_STAY.RAW.CSV_FF);
```
</details>

## Clean and standardize with CoCo

Now we turn three messy exports into one governed campaign list. This is the step that usually takes a marketing ops analyst a day of spreadsheet work. With CoCo, you describe the rules in plain language and review the SQL it writes.

The rules:

- **One vocabulary.** Every channel, region, and audience maps to a single standard label.
- **Real dates and numbers.** Three date formats become real `DATE` values; `"$65,000"` becomes `65000.00`.
- **No duplicates.** A campaign that appears in more than one system (or twice in the same export) is kept once, preferring the system of record (ad platforms or the email & SMS platform) over the CRM copy.
- **No cancelled campaigns.** A cancelled CRM campaign shouldn't show up on anyone's calendar.

The output is another **Iceberg** table, `CURATED.CAMPAIGNS`, so the clean data is just as open as the raw data.

### STEP 2 — Let CoCo find the mess

Before building anything, ask CoCo to compare the three sources. It will query the raw tables itself and show you how differently each system describes the *same* things. The standard values in the prompt are your team's business vocabulary, so every system ends up speaking the same language.

> *"Compare how the three tables in MERIDIAN_STAY.RAW label channel, region, and audience. Map every label to one of these standard values:*
> - *Channel: Paid Social, Paid Search, Display, Email, SMS*
> - *Region: North America, EMEA, APAC, LATAM, Global*
> - *Audience: Leisure Travelers, Business Travelers, Families, Loyalty Members, Meeting Planners, Wellness Seekers, All Guests"*

There's nothing to run in a cell for this step. Read CoCo's mapping and check a few labels against the table below.

<details>
<summary>Expected mapping (check against this)</summary>

| Standard value | Raw labels |
|---|---|
| Paid Social | `fb_ads`, `paid_social`, `Social Media` |
| Paid Search | `google_ads`, `sem`, `Search Engine Marketing` |
| Display | `programmatic`, `Display Advertising` |
| Email | `Email`, `Email Marketing` |
| SMS | `SMS` |
| North America | `NA`, `north america`, `US & Canada`, `N. America` |
| EMEA | `EMEA`, `Europe/ME/Africa` |
| APAC | `apac`, `Asia Pacific`, `Asia-Pacific` |
| LATAM | `latam` |
| Global | `global`, `Worldwide` |
| Leisure Travelers | `leisure`, `leisure_travelers`, `Leisure Segment`, `Leisure` |
| Business Travelers | `business`, `biz_travel`, `Corporate Segment`, `Business` |
| Families | `families`, `family`, `Family Segment`, `Family Travelers` |
| Loyalty Members | `loyalty`, `Meridian Rewards Members`, `Rewards Members` |
| Meeting Planners | `meeting_planners` |
| Wellness Seekers | `wellness` |
| All Guests | `all`, `All` |

CoCo should also notice the three date formats (`2026-10-05`, `10/12/2026`, `Oct 05, 2026`) and the text budgets (`"$65,000"`).
</details>

### STEP 3 — Build the unified CAMPAIGNS table

In the **same CoCo conversation**, send this prompt. CoCo reuses the mapping from STEP 2 and writes the SQL. Your SQL may not match the expected output word for word, and that's fine; the **check cell** below is what tells you it's right. Click **Allow** to run it.

> *"Using that mapping, create a Snowflake-managed Iceberg table MERIDIAN_STAY.CURATED.CAMPAIGNS (EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED') that combines the three raw tables into one campaign list:*
> - *Columns: campaign_id, campaign_name, channel, region, audience, start_date, end_date, budget_usd, owner_team, status, source_system*
> - *Convert dates and budgets to real dates and numbers*
> - *Drop cancelled campaigns*
> - *Remove duplicates with the same name (ignoring case and extra spaces) and dates, keeping the ad platform or email & SMS copy over the CRM copy"*

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY.CURATED.CAMPAIGNS
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
AS
WITH unioned AS (
    SELECT ad_campaign_id AS campaign_id, campaign_name, platform_channel AS channel_raw, geo AS region_raw,
           target_audience AS audience_raw,
           TRY_TO_DATE(flight_start, 'YYYY-MM-DD') AS start_date,
           TRY_TO_DATE(flight_end, 'YYYY-MM-DD') AS end_date,
           TRY_TO_NUMBER(REPLACE(REPLACE(budget, '$', ''), ',', ''), 12, 2) AS budget_usd,
           owner AS owner_team, 'Active' AS status, 'Paid Media' AS source_system, 1 AS source_priority
    FROM MERIDIAN_STAY.RAW.PAID_MEDIA_EXPORT
    UNION ALL
    SELECT program_id, program_name, channel_type, region_name, segment,
           TRY_TO_DATE(send_start, 'MM/DD/YYYY'), TRY_TO_DATE(send_end, 'MM/DD/YYYY'),
           TRY_TO_NUMBER(budget_usd, 12, 2), team, 'Active', 'Email & SMS', 1
    FROM MERIDIAN_STAY.RAW.EMAIL_SMS_EXPORT
    UNION ALL
    SELECT crm_campaign_id, campaign_title, campaign_type, territory, audience_group,
           TRY_TO_DATE(start_date, 'MON DD, YYYY'), TRY_TO_DATE(end_date, 'MON DD, YYYY'),
           TRY_TO_NUMBER(budgeted_cost, 12, 2), 'Unassigned', status, 'CRM', 2
    FROM MERIDIAN_STAY.RAW.CRM_CAMPAIGNS_EXPORT
)
SELECT
    campaign_id::STRING AS campaign_id,
    TRIM(campaign_name)::STRING AS campaign_name,
    CASE LOWER(TRIM(channel_raw))
        WHEN 'fb_ads' THEN 'Paid Social' WHEN 'meta' THEN 'Paid Social' WHEN 'paid_social' THEN 'Paid Social'
        WHEN 'social media' THEN 'Paid Social'
        WHEN 'google_ads' THEN 'Paid Search' WHEN 'sem' THEN 'Paid Search' WHEN 'search engine marketing' THEN 'Paid Search'
        WHEN 'display' THEN 'Display' WHEN 'programmatic' THEN 'Display' WHEN 'display advertising' THEN 'Display'
        WHEN 'email' THEN 'Email' WHEN 'email marketing' THEN 'Email'
        WHEN 'sms' THEN 'SMS' WHEN 'text messaging' THEN 'SMS'
    END::STRING AS channel,
    CASE LOWER(TRIM(region_raw))
        WHEN 'na' THEN 'North America' WHEN 'north america' THEN 'North America' WHEN 'us & canada' THEN 'North America'
        WHEN 'n. america' THEN 'North America'
        WHEN 'emea' THEN 'EMEA' WHEN 'europe/me/africa' THEN 'EMEA' WHEN 'europe' THEN 'EMEA'
        WHEN 'apac' THEN 'APAC' WHEN 'asia pacific' THEN 'APAC' WHEN 'asia-pacific' THEN 'APAC'
        WHEN 'latam' THEN 'LATAM' WHEN 'latin america' THEN 'LATAM'
        WHEN 'global' THEN 'Global' WHEN 'worldwide' THEN 'Global'
    END::STRING AS region,
    CASE LOWER(TRIM(audience_raw))
        WHEN 'leisure' THEN 'Leisure Travelers' WHEN 'leisure_travelers' THEN 'Leisure Travelers' WHEN 'leisure segment' THEN 'Leisure Travelers'
        WHEN 'business' THEN 'Business Travelers' WHEN 'biz_travel' THEN 'Business Travelers' WHEN 'corporate segment' THEN 'Business Travelers'
        WHEN 'families' THEN 'Families' WHEN 'family' THEN 'Families' WHEN 'family segment' THEN 'Families' WHEN 'family travelers' THEN 'Families'
        WHEN 'loyalty' THEN 'Loyalty Members' WHEN 'meridian rewards members' THEN 'Loyalty Members' WHEN 'rewards members' THEN 'Loyalty Members'
        WHEN 'meeting_planners' THEN 'Meeting Planners' WHEN 'event planners' THEN 'Meeting Planners'
        WHEN 'wellness' THEN 'Wellness Seekers' WHEN 'wellness segment' THEN 'Wellness Seekers'
        WHEN 'all' THEN 'All Guests'
    END::STRING AS audience,
    start_date,
    end_date,
    budget_usd::NUMBER(12,2) AS budget_usd,
    owner_team::STRING AS owner_team,
    status::STRING AS status,
    source_system::STRING AS source_system
FROM unioned
WHERE status NOT ILIKE 'cancel%'
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY LOWER(TRIM(campaign_name)), start_date, end_date
    ORDER BY source_priority, campaign_id) = 1;
```
</details>

### Check the result

Run this check. You should see **27 campaigns**, **0 unmapped values**, **5 channels**, **5 regions**, and **7 audiences** (33 raw rows minus 5 duplicates and 1 cancelled campaign).

If your numbers are different, ask CoCo to fix it rather than editing the SQL yourself:
- `unmapped_values` above 0: *"Which rows in MERIDIAN_STAY.CURATED.CAMPAIGNS have a NULL channel, region, audience, date, or budget? Fix the mapping."*
- More than 27 campaigns: *"Which campaigns appear more than once in MERIDIAN_STAY.CURATED.CAMPAIGNS? Fix the duplicate removal."*

If you're short on time, paste the expected output from STEP 3 into the empty cell above and run it.

In [ ]:
SELECT
    COUNT(*) AS campaigns,
    COUNT_IF(channel IS NULL OR region IS NULL OR audience IS NULL
             OR start_date IS NULL OR end_date IS NULL OR budget_usd IS NULL) AS unmapped_values,
    COUNT(DISTINCT channel) AS channels,
    COUNT(DISTINCT region) AS regions,
    COUNT(DISTINCT audience) AS audiences
FROM MERIDIAN_STAY.CURATED.CAMPAIGNS;

### STEP 4 — Find every audience collision

A **collision** is two different campaigns that target the same audience in the same region on overlapping dates — the situation that caused last quarter's problem.

Send this prompt to CoCo. Compare CoCo's output against the expected output below. Column names may differ slightly, and that's fine. Click **Allow** to run it.

> *"Create a Snowflake-managed Iceberg table MERIDIAN_STAY.CURATED.CAMPAIGN_COLLISIONS (EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED') listing every pair of campaigns in CURATED.CAMPAIGNS that target the same region and audience on overlapping dates. List each pair once, with both campaign names, the overlap dates, and their combined budget."*

<details>
<summary>Expected output (check against this)</summary>

```sql
CREATE OR REPLACE ICEBERG TABLE MERIDIAN_STAY.CURATED.CAMPAIGN_COLLISIONS
  CATALOG = 'SNOWFLAKE'
  EXTERNAL_VOLUME = 'SNOWFLAKE_MANAGED'
AS
SELECT
    a.campaign_id   AS campaign_a_id,
    a.campaign_name AS campaign_a_name,
    a.channel       AS campaign_a_channel,
    b.campaign_id   AS campaign_b_id,
    b.campaign_name AS campaign_b_name,
    b.channel       AS campaign_b_channel,
    a.region,
    a.audience,
    GREATEST(a.start_date, b.start_date) AS overlap_start,
    LEAST(a.end_date, b.end_date)        AS overlap_end,
    DATEDIFF('day', GREATEST(a.start_date, b.start_date), LEAST(a.end_date, b.end_date)) + 1 AS overlap_days,
    a.budget_usd + b.budget_usd          AS combined_budget_usd
FROM MERIDIAN_STAY.CURATED.CAMPAIGNS a
JOIN MERIDIAN_STAY.CURATED.CAMPAIGNS b
  ON  a.region = b.region
  AND a.audience = b.audience
  AND a.campaign_id < b.campaign_id
  AND a.start_date <= b.end_date
  AND b.start_date <= a.end_date;
```
</details>

### Check the result

You should see **9 rows** (`total_collisions` = 9), with **$557,500** of combined budget across them. Notice the cancelled *Columbus Day Weekend Blitz* isn't among them — without the clean-up step, it would have raised two false alarms.

If you see 18 rows, each pair is listed twice; ask CoCo to *"list each pair only once."*

In [ ]:
SELECT COUNT(*) OVER () AS total_collisions, *
FROM MERIDIAN_STAY.CURATED.CAMPAIGN_COLLISIONS;

## See it: the live campaign calendar

The clean `CAMPAIGNS` table now powers a shared **campaign calendar**, a Streamlit app that's already in this repo.

1. In the Workspace file explorer, open **`campaign_calendar/streamlit_app.py`**.
2. If the Workspace asks to create Streamlit configuration files for the app, accept.
3. Click **Run**. The app reads `MERIDIAN_STAY.CURATED.CAMPAIGNS` and draws every campaign on a month grid, color-coded by channel. Pick **November 2026** and **December 2026** to see how crowded the holiday season is.

### STEP 5 — Make collisions impossible to miss (with CoCo)

With `streamlit_app.py` open, send this prompt to CoCo:

> *"Update this calendar to also load CURATED.CAMPAIGN_COLLISIONS. Outline colliding campaigns in red on the days they overlap, add a metric for collisions this month, and list them in a table below the calendar."*

Review the diff CoCo proposes, accept it, and click **Run** again. In **October 2026** you should see the North America business-traveler pile-up: *Business Travel Q4 Push*, *Road Warrior Rewards*, and *Amex Travel Partner Promo* all outlined in red.

## Data is ready → on to CoWork (in the UI)

You now have:

- `RAW.*` — the three exports, exactly as they arrived, in open Iceberg tables
- `CURATED.CAMPAIGNS` — 27 clean, deduplicated campaigns in one vocabulary (Iceberg)
- `CURATED.CAMPAIGN_COLLISIONS` — the 9 audience collisions (Iceberg)
- A live campaign calendar anyone on the team can open

The last step happens in the Snowsight UI (no code): create a **Semantic View** over the two `CURATED` tables, create a **Cortex Agent** backed by it, and ask your questions in **Snowflake CoWork**. Follow the **"Ask It With CoWork"** section of the guide.

## Teardown

Once you're done with the lab, run the cell below to remove everything it created.

> **Note:** Dropping `MERIDIAN_STAY` cascades to everything inside it: Iceberg tables, stage, file format, semantic view, and Cortex Agent.

In [ ]:
-- TEARDOWN: remove all lab objects
USE ROLE ACCOUNTADMIN;

-- Drops all schemas, Iceberg tables, the stage, the semantic view, and the Cortex Agent
DROP DATABASE IF EXISTS MERIDIAN_STAY;

-- Removes the Git API integration created for the workspace (update the name if you used a different one)
DROP API INTEGRATION IF EXISTS GITHUB_MERIDIAN_LAB;